In [20]:
import torch
import tiktoken
import torch.nn as nn
import torch.nn.functional as F
import math
if torch.mps.is_available():
    device = "mps"
else:
    raise Exception("no mps")

encoder = tiktoken.encoding_for_model("gpt-2")
print(encoder)


<Encoding 'gpt2'>


In [2]:
data = ""
with open("./data/verdict.txt") as fp:
    data = fp.read()

In [ ]:
CONTEXT_LENGTH = 64
VOCAB_SIZE = encoder.n_vocab
BATCH_SIZE = 32
EMBEDDING_DIMENSION = 128
encodings = encoder.encode(data)
print(len(encodings))
SIZE = (len(encodings)-CONTEXT_LENGTH-1, CONTEXT_LENGTH)

5145


In [ ]:

xs = torch.empty(size=SIZE, dtype=torch.int32)
ys = torch.empty(size=SIZE, dtype=torch.int32)

for i in range((len(encodings)-CONTEXT_LENGTH-1)):
    x = torch.tensor(encodings[i: i+CONTEXT_LENGTH])
    y = torch.tensor(encodings[i+1: i+1+CONTEXT_LENGTH])
    if len(x) != CONTEXT_LENGTH:
        raise Exception(f"len(x) should be {CONTEXT_LENGTH}")
    if len(y) != CONTEXT_LENGTH:
        raise Exception(f"len(y) should be {CONTEXT_LENGTH}")    
    xs[i] = x
    ys[i] = y


In [15]:
assert xs.shape == SIZE
assert ys.shape == SIZE

from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(xs, ys)
dataloader = DataLoader(dataset=dataset, shuffle=False, batch_size=BATCH_SIZE)

torch.Size([32, 64])

In [ ]:
class CasualAttention(nn.Module):
    def __init__(self, head_dim:int, embedding_dimension:int, mask:torch.Tensor) -> None:
        super().__init__()
        self.head_dim = head_dim
        self.mask = mask
        self.query = nn.Linear(embedding_dimension, head_dim, bias=False)
        self.key = nn.Linear(embedding_dimension, head_dim, bias=False)
        self.value = nn.Linear(embedding_dimension, head_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # shape of x here is (B, CONTENT_LENGTH, EMBEDDING_DIM)
        q, k, v = self.query(x), self.key(x), self.value(x) 

        attn_scores = (q @ k.tanspose(-2, -1)) + self.mask # shape (B, CONTENT_LENGTH, CONTENT_LENGTH)

        return F.softmax(attn_scores/math.sqrt(self.head_dim), dim=-1) @ v

        


class MultiHeadCasualAttention(nn.Module):
    def __init__(self, num_heads: int, embedding_dimension:int, context_length:int) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.embedding_dimension = embedding_dimension
        self.context_length = context_length

        assert self.embedding_dimension % self.num_heads == 0, "embedding dimension should be divisible by num heads"
        self.head_dim =  self.embedding_dimension // self.num_heads

        self.mask = torch.triu(torch.ones(size=(self.context_length, self.context_length)), diagonal=1)
        self.mask = torch.where(self.mask == 1, -math.inf, 0)
        self.maksed_attns = [
            CasualAttention(head_dim=self.head_dim, embedding_dimension=self.embedding_dimension, mask=self.mask) 
            for _ in range(self.num_heads)
            ]


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # input is (B, context_length, embed_dim)
        # and output is (B, context_length, embed_dim)
        return torch.concat([attn(x) for attn in self.maksed_attns], dim=-1)
        

In [ ]:


class AbbyGPT(nn.Module):
    def __init__(self, vocab_size:int, embedding_dimension:int, context_length: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dimension = embedding_dimension
        self.context_length = context_length
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dimension)
        self.multi_head_attn = MultiHeadCasualAttention(num_heads=4, embedding_dimension=self.embedding_dimension, context_length=self.context_length)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x is (batch_size, context_length)
        input_tensor = x
        x = self.embedding_layer(x) # (B, context_length, embed_dim)
        x = self.multi_head_attn(x)
        return x
